In [19]:
import pandas as pd

chexpert = pd.read_csv('mimic-cxr-2.0.0-chexpert.csv.gz')
metadata = pd.read_csv('mimic-cxr-2.0.0-metadata.csv.gz')
split = pd.read_csv('mimic-cxr-2.0.0-split.csv.gz')

In [20]:
target_labels = [
"Cardiomegaly",
"Edema",
"Pleural Effusion"
]

df = chexpert.copy()
# 1. 세 소견 모두 NaN인 검사 제외

all_nan = df[target_labels].isna().all(axis=1)
df = df[~all_nan].copy()

# 2. 나머지 NaN을 0으로 대체 (-1은 유지)
df[target_labels] = df[target_labels].fillna(0)

# 3. 전처리 결과 확인
print(f"전처리 후 검사 수: {len(df):,}")

for label in target_labels:
    print(f"\n===== {label} =====")
    print(df[label].value_counts(dropna=False))

전처리 후 검사 수: 122,564

===== Cardiomegaly =====
Cardiomegaly
 0.0    71676
 1.0    44845
-1.0     6043
Name: count, dtype: int64

===== Edema =====
Edema
 0.0    82372
 1.0    27018
-1.0    13174
Name: count, dtype: int64

===== Pleural Effusion =====
Pleural Effusion
 0.0    62450
 1.0    54300
-1.0     5814
Name: count, dtype: int64


In [21]:
target_labels = [
    "Cardiomegaly",
    "Edema",
    "Pleural Effusion"
]

# 1. 전처리된 검사 데이터 준비
df = chexpert.copy()

all_nan = df[target_labels].isna().all(axis=1)
df = df[~all_nan].copy()

df[target_labels] = df[target_labels].fillna(0)

# 2. Metadata 병합
image_df = metadata.merge(
    df[["subject_id", "study_id"] + target_labels],
    on=["subject_id", "study_id"],
    how="inner",
    validate="many_to_one"
)

# 3. Split 병합
image_df = image_df.merge(
    split[["dicom_id", "subject_id", "study_id", "split"]],
    on=["dicom_id", "subject_id", "study_id"],
    how="inner",
    validate="one_to_one"
)

# 4. AP, PA 이미지만 선택
image_df = image_df[
    image_df["ViewPosition"].isin(["AP", "PA"])
].copy()

# 5. 촬영 방향 및 Split별 이미지 수
summary = image_df.groupby(
    ["split", "ViewPosition"]
).size().unstack(fill_value=0)

summary["Total"] = summary.sum(axis=1)

display(summary)

# 6. Split별 세 소견 양성 이미지 수
positive_count = image_df.groupby("split")[target_labels].agg(
    lambda x: (x == 1).sum()
)

display(positive_count)

ViewPosition,AP,PA,Total
split,,,
test,1744,540,2284
train,93950,32256,126206
validate,819,258,1077


,Cardiomegaly,Edema,Pleural Effusion
split,,,
test,896,726,1095
train,46375,28341,56121
validate,402,264,505


In [22]:
target_labels = [
    "Cardiomegaly",
    "Edema",
    "Pleural Effusion"
]

results = []

for view in ["AP", "PA"]:
    subset = image_df[image_df["ViewPosition"] == view]

    for label in target_labels:
        positive = (subset[label] == 1).sum()
        negative = (subset[label] == 0).sum()
        uncertain = (subset[label] == -1).sum()

        results.append({
            "ViewPosition": view,
            "Finding": label,
            "Positive": positive,
            "Negative": negative,
            "Uncertain": uncertain,
            "Positive (%)": round(
                positive / (positive + negative) * 100, 2
            ) if positive + negative > 0 else 0
        })

view_distribution = pd.DataFrame(results)

display(view_distribution)

,ViewPosition,Finding,Positive,Negative,Uncertain,Positive (%)
0,AP,Cardiomegaly,37842,53480,5191,41.44
1,AP,Edema,26202,58619,11692,30.89
2,AP,Pleural Effusion,45657,46051,4805,49.79
3,PA,Cardiomegaly,9831,21997,1226,30.89
4,PA,Edema,3129,27373,2552,10.26
5,PA,Pleural Effusion,12064,19593,1397,38.11


In [23]:
# Train 데이터만 선택
train_df = image_df[image_df["split"] == "train"].copy()

# AP, PA 각각 추출
ap_sample = train_df[
    train_df["ViewPosition"] == "AP"
].sample(n=7444, random_state=42)

pa_sample = train_df[
    train_df["ViewPosition"] == "PA"
].sample(n=2556, random_state=42)

# 두 데이터 합치기
train_sample = pd.concat(
    [ap_sample, pa_sample],
    ignore_index=True
)

print(f"선정된 이미지 수: {len(train_sample):,}")
print(train_sample["ViewPosition"].value_counts())

선정된 이미지 수: 10,000
ViewPosition
AP    7444
PA    2556
Name: count, dtype: int64


In [24]:
target_labels = [
    "Cardiomegaly",
    "Edema",
    "Pleural Effusion"
]

label_result = pd.DataFrame({
    "Positive": (train_sample[target_labels] == 1).sum(),
    "Negative": (train_sample[target_labels] == 0).sum(),
    "Uncertain": (train_sample[target_labels] == -1).sum()
})

display(label_result)

,Positive,Negative,Uncertain
Cardiomegaly,3775,5729,496
Edema,2224,6696,1080
Pleural Effusion,4488,5070,442


In [25]:
# 이미지별 양성 소견 개수
positive_count = (
    train_sample[target_labels] == 1
).sum(axis=1)

# 양성 소견 개수별 분포
combination_result = positive_count.value_counts().reindex(
    [0, 1, 2, 3],
    fill_value=0
).sort_index()

combination_result.index = [
    "양성 0개",
    "양성 1개",
    "양성 2개",
    "양성 3개"
]

display(combination_result)

print(f"복합 소견(2개 이상 양성): {(positive_count >= 2).sum():,}")
print(f"세 소견 모두 양성: {(positive_count == 3).sum():,}")

양성 0개    2625
양성 1개    4827
양성 2개    1984
양성 3개     564
Name: count, dtype: int64

복합 소견(2개 이상 양성): 2,548
세 소견 모두 양성: 564


In [26]:
print(
    "고유 환자 수:",
    train_sample["subject_id"].nunique()
)

print(
    "고유 검사 수:",
    train_sample["study_id"].nunique()
)

print(
    "고유 이미지 수:",
    train_sample["dicom_id"].nunique()
)

고유 환자 수: 7204
고유 검사 수: 9932
고유 이미지 수: 10000


In [27]:
# 세 소견 모두 0인 이미지
all_negative = train_sample[target_labels].eq(0).all(axis=1)

# 양성 소견은 없지만 -1이 포함된 이미지
no_positive = train_sample[target_labels].ne(1).all(axis=1)
uncertain_only = no_positive & train_sample[target_labels].eq(-1).any(axis=1)

print(f"세 소견 모두 음성: {all_negative.sum():,}")
print(f"양성 없이 불확실 포함: {uncertain_only.sum():,}")

세 소견 모두 음성: 1,759
양성 없이 불확실 포함: 866


In [28]:
train_patients = set(train_sample["subject_id"])

val_patients = set(
    image_df.loc[image_df["split"] == "validate", "subject_id"]
)

test_patients = set(
    image_df.loc[image_df["split"] == "test", "subject_id"]
)

print("Train-Val 중복:", len(train_patients & val_patients))
print("Train-Test 중복:", len(train_patients & test_patients))
print("Val-Test 중복:", len(val_patients & test_patients))

Train-Val 중복: 0
Train-Test 중복: 0
Val-Test 중복: 0


In [31]:
# 최초 샘플링한 Train 10,000장 사용
train_final = train_sample.copy()

# 최종 데이터 검증
assert len(train_final) == 10000
assert train_final["dicom_id"].is_unique

all_uncertain = train_final[target_labels].eq(-1).all(axis=1)

print(f"최종 Train 이미지 수: {len(train_final):,}")

최종 Train 이미지 수: 10,000


In [29]:
from pathlib import Path

# 저장할 컬럼
columns = [
    "subject_id",
    "study_id",
    "dicom_id",
    "ViewPosition",
    "split",
    "Cardiomegaly",
    "Edema",
    "Pleural Effusion"
]

# Validate / Test 추출
val_final = image_df[
    image_df["split"] == "validate"
][columns].copy()

test_final = image_df[
    image_df["split"] == "test"
][columns].copy()

# CSV 저장
save_dir = Path("../data/processed")
save_dir.mkdir(parents=True, exist_ok=True)

train_final[columns].to_csv(save_dir / "train.csv", index=False)
val_final.to_csv(save_dir / "val.csv", index=False)
test_final.to_csv(save_dir / "test.csv", index=False)

# 결과 확인
print(f"Train: {len(train_final):,}")
print(f"Validate: {len(val_final):,}")
print(f"Test: {len(test_final):,}")
print("CSV 저장 완료")

Train: 10,000
Validate: 1,077
Test: 2,284
CSV 저장 완료


## 데이터셋 선정 결과 요약

### 1. 레이블 전처리

심비대(Cardiomegaly), 폐부종(Edema), 흉수(Pleural Effusion) 세 소견을 대상으로 레이블 전처리를 진행하였다.

| 레이블 | 처리 기준 |
|---|---|
| 양성(1) | 유지 |
| 음성(0) | 유지 |
| 불확실(-1) | 유지 |
| 결측(NaN) | 0으로 대체 |
| 세 소견 모두 NaN | 해당 검사 제외 |

- 전처리 전 검사 수: 227,827건
- 전처리 후 검사 수: **122,564건**
- 불확실 레이블(-1)은 U-MultiClass 방식에 따라 별도 클래스로 학습할 예정이다.
- NaN을 0으로 대체한 것은 해당 소견을 음성으로 가정한 것이며, 실제 음성이 확인된 것은 아니다.
- `No Finding`은 학습 레이블로 사용하지 않는다.

### 2. 이미지 데이터셋 구성

CheXpert, Metadata, Split 파일을 병합하고 AP 및 PA 촬영 이미지만 선정하였다.

| Split | AP | PA | Total |
|---|---:|---:|---:|
| Train | 93,950 | 32,256 | 126,206 |
| Validate | 819 | 258 | 1,077 |
| Test | 1,744 | 540 | 2,284 |
| **합계** | **96,513** | **33,054** | **129,567** |

- AP와 PA 촬영 이미지를 모두 모델 학습에 활용하기로 결정하였다.
- MIMIC-CXR-JPG에서 제공하는 공식 Train / Validate / Test 분할을 유지하였다.
- Train에서는 일부 이미지를 샘플링하고, Validate와 Test는 선정 조건을 만족하는 전체 이미지를 사용한다.

### 3. 학습 이미지 샘플링

전체 Train 데이터 126,206장 중 10,000장을 무작위 추출하였다.

| 촬영 방향 | 선정 이미지 수 |
|---|---:|
| AP | 7,444 |
| PA | 2,556 |
| **합계** | **10,000** |

- 기존 Train 데이터의 AP/PA 비율을 유지하였다.
- `random_state=42`를 적용하여 재현 가능한 샘플링을 수행하였다.
- 불확실 레이블(-1)은 제거하지 않고 그대로 유지하였다.
- 세 소견 모두 불확실(-1)한 이미지 4장도 학습 데이터에 포함하였다.

최종적으로 **Train 이미지 10,000장**을 선정하였다.

### 4. 선정 데이터의 레이블 분포

최종 선정한 Train 10,000장의 소견별 레이블 분포는 다음과 같다.

| 소견 | 양성(1) | 음성(0) | 불확실(-1) |
|---|---:|---:|---:|
| Cardiomegaly | 3,775 | 5,729 | 496 |
| Edema | 2,224 | 6,696 | 1,080 |
| Pleural Effusion | 4,488 | 5,070 | 442 |

- 세 소견 모두 양성 사례가 2,000장 이상 확보되었다.
- 폐부종은 다른 소견에 비해 양성 사례가 적고 불확실 레이블 비율이 높게 나타났다.
- 음성(0)에는 기존 음성 레이블과 NaN에서 대체된 값이 함께 포함되어 있다.
- 불확실(-1) 레이블은 U-MultiClass 학습에서 별도 클래스로 활용한다.

### 5. 복합 소견 분포

선정한 Train 10,000장에서 양성(1)으로 확인된 소견의 개수를 기준으로 복합 소견 분포를 분석하였다.

| 양성 소견 수 | 이미지 수 | 비율 |
|---|---:|---:|
| 0개 | 2,625 | 26.25% |
| 1개 | 4,827 | 48.27% |
| 2개 | 1,984 | 19.84% |
| 3개 | 564 | 5.64% |

- 두 가지 이상 동시 양성인 이미지: **2,548장 (25.48%)**
- 세 소견 모두 양성인 이미지: **564장 (5.64%)**
- 세 소견 모두 음성인 이미지: 1,759장
- 양성 없이 불확실 레이블이 포함된 이미지: 866장
- 세 소견 모두 불확실한 이미지: 4장

양성 0개에 해당하는 이미지에는 불확실(-1) 레이블이 포함된 사례도 존재하므로, 세 소견 모두 음성인 이미지와 구분해야 한다.

세 소견 모두 불확실한 이미지 4장은 U-MultiClass 방식에서 학습 가능한 데이터이므로 제외하지 않았다.

### 6. 데이터 중복 및 구성 확인

선정한 Train 10,000장을 기준으로 고유 환자·검사·이미지 수를 확인하였다.

| 구분 | 수량 |
|---|---:|
| 고유 환자 수 | 7,204명 |
| 고유 검사 수 | 9,932건 |
| 고유 이미지 수 | 10,000장 |

- 이미지 식별자(`dicom_id`) 기준 중복 여부를 확인하였다.
- 동일 검사에 여러 이미지가 포함될 수 있으므로 검사 수와 이미지 수를 구분하였다.
- 공식 Split을 유지하여 Train / Validate / Test 데이터를 구성하였다.

### 7. 최종 데이터셋 구성

| 구분 | AP | PA | Total |
|---|---:|---:|---:|
| Train | 7,444 | 2,556 | 10,000 |
| Validate | 819 | 258 | 1,077 |
| Test | 1,744 | 540 | 2,284 |
| **합계** | **10,007** | **3,354** | **13,361** |

- Train은 기존 AP/PA 비율을 유지한 10,000장으로 구성하였다.
- Validate와 Test는 공식 Split을 유지하고 별도 무작위 샘플링 없이 사용한다.
- 각 데이터셋을 `train.csv`, `val.csv`, `test.csv`로 저장하도록 구성하였다.
- 양성(1), 음성(0), 불확실(-1) 레이블을 모두 보존하였다.

**모델 개발 방향**

- 모델: DenseNet121
- 학습 방식: U-MultiClass
- 예측 대상: 심비대, 폐부종, 흉수
- 분류 구조: 각 소견별 음성·양성·불확실 3클래스 분류
- 출력 구조: 총 9개 출력값 (3개 소견 × 3개 클래스)
- 손실 함수: 소견별 Cross-Entropy Loss
- 학습 시 불확실 레이블(-1)은 클래스 인덱스로 변환하여 활용

### 8. 결론

MIMIC-CXR-JPG 데이터에서 심비대·폐부종·흉수 소견 탐지를 위한 레이블 전처리 및 이미지 선정 기준을 수립하였다.

CheXpert, Metadata, Split 데이터를 병합하여 AP/PA 이미지를 추출하고, 촬영 방향별 분포를 고려하여 Train 10,000장, Validate 1,077장, Test 2,284장으로 구성된 총 **13,361장의 이미지 데이터셋**을 선정하였다.

세 소견의 양성·음성 및 복합 소견 사례가 확보되었으며, 불확실 레이블을 별도 클래스로 활용하는 U-MultiClass 방식으로 모델을 학습할 예정이다.

다만 NaN을 음성으로 대체한 전처리 방식의 한계와 자동 추출된 불확실 레이블의 특성을 고려하여 모델 성능을 해석할 필요가 있다.